<a href="https://colab.research.google.com/github/muhyassin09/yasinnn/blob/main/medical-survival-analysis/scripts/01_data-preprocessing-and-table-baseline-characteristics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HEART FAILURE CLINICAL RECORDS — BASELINE CHARACTERISTICS (TABLE 1)

Environment Setup

In [1]:
!pip install -q tableone ucimlrepo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.0 MB/s eta 0:00:00


In [2]:
import os
import zipfile
import urllib.request
import pandas as pd
from tableone import TableOne

INPUT DATA

In [7]:
heart_failure_data = pd.read_csv('heart_failure_clinical_records_dataset.csv')

In [16]:
heart_failure_data.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


In [17]:
heart_failure_data = heart_failure_data[expected_columns].apply(pd.to_numeric)

In [18]:
n_deaths = heart_failure_data["DEATH_EVENT"].sum()
n_total = len(heart_failure_data)

print(f"Loaded cohort: N = {n_total} rows | Deaths = {n_deaths} ({100 * n_deaths / n_total:.1f}%)")

Loaded cohort: N = 299 rows | Deaths = 96 (32.1%)


In [19]:
heart_failure_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       299 non-null    float64
 1   anaemia                   299 non-null    int64  
 2   creatinine_phosphokinase  299 non-null    int64  
 3   diabetes                  299 non-null    int64  
 4   ejection_fraction         299 non-null    int64  
 5   high_blood_pressure       299 non-null    int64  
 6   platelets                 299 non-null    float64
 7   serum_creatinine          299 non-null    float64
 8   serum_sodium              299 non-null    int64  
 9   sex                       299 non-null    int64  
 10  smoking                   299 non-null    int64  
 11  time                      299 non-null    int64  
 12  DEATH_EVENT               299 non-null    int64  
dtypes: float64(3), int64(10)
memory usage: 30.5 KB


VARIABLE LABELING

In [65]:
categorical_columns = [
    "anaemia", "diabetes", "high_blood_pressure",
    "sex", "smoking", "DEATH_EVENT"
]

In [52]:
label_maps = {
    "anaemia":             {0: "No", 1: "Yes"},
    "diabetes":            {0: "No", 1: "Yes"},
    "high_blood_pressure": {0: "No", 1: "Yes"},
    "smoking":             {0: "No", 1: "Yes"},
    "sex":                 {0: "Female", 1: "Male"},
    "DEATH_EVENT":         {0: "Survived", 1: "Deceased"},
}

In [54]:
heart_failure_tbl1 = heart_failure_data.copy()
for col, mapping in label_maps.items():
    if col == groupby_column:
        heart_failure_tbl1[col] = heart_failure_tbl1[col].map(mapping).astype("object")
    else:
        heart_failure_tbl1[col] = heart_failure_tbl1[col].map(mapping).astype("category")

In [50]:
display_labels = {
    "age":                      "Age (years)",
    "anaemia":                  "Anaemia",
    "creatinine_phosphokinase": "Creatine Phosphokinase (mcg/L)",
    "diabetes":                 "Diabetes Mellitus",
    "ejection_fraction":        "Ejection Fraction (%)",
    "high_blood_pressure":      "Hypertension",
    "platelets":                "Platelets (kiloplatelets/mL)",
    "serum_creatinine":         "Serum Creatinine (mg/dL)",
    "serum_sodium":             "Serum Sodium (mEq/L)",
    "sex":                      "Sex",
    "smoking":                  "Current Smoker",
    "time":                     "Follow-up Time (days)",
}

TABLE 1: BASELINE CHARACTERISTICS

In [61]:
groupby_column = "DEATH_EVENT"

continuous_columns = [
    "age", "creatinine_phosphokinase", "ejection_fraction",
    "platelets", "serum_creatinine", "serum_sodium", "time"
]

In [62]:
nonnormal_override = [
    "creatinine_phosphokinase", "platelets", "serum_creatinine", "time"
]

In [66]:
table1 = TableOne(
    data=heart_failure_tbl1,
    columns=continuous_columns + categorical_columns,
    categorical=categorical_columns,
    groupby=groupby_column,
    nonnormal=nonnormal_override,   # forces median [IQR] + Mann-Whitney U for these
    pval=True,                      # compute and display group-comparison p-values
    pval_test_name=True,            # show which statistical test was used per row
    htest_name=True,
    overall=True,                   # include an "Overall" column alongside the two groups
    rename=display_labels,          # apply publication-style row labels
    label_suffix=True,              # append units/labels defined above to row headers
)

In [67]:
print(table1.tabulate(tablefmt="fancy_grid"))

╒════════════════════════════════════════════════╤══════════╤═══════════╤══════════════════════════════╤══════════════════════════════╤══════════════════════════════╤═══════════╤════════════════╕
│                                                │          │ Missing   │ Overall                      │ Deceased                     │ Survived                     │ P-Value   │ Test           │
╞════════════════════════════════════════════════╪══════════╪═══════════╪══════════════════════════════╪══════════════════════════════╪══════════════════════════════╪═══════════╪════════════════╡
│ n                                              │          │           │ 299                          │ 96                           │ 203                          │           │                │
├────────────────────────────────────────────────┼──────────┼───────────┼──────────────────────────────┼──────────────────────────────┼──────────────────────────────┼───────────┼────────────────┤
│ Age (years), mean 